In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
import json
import os

# CONFIGURAZIONE (Sostituisci se il nome è diverso)
FILE_GREEN = "/home/giacomo/LLMs_For_Green_Code_Refactoring/data/processed/green/swe_perf_green_k1.json"

def process_green_dataset(filepath):
    if not os.path.exists(filepath):
        print(f"❌ File non trovato: {filepath}")
        return None

    with open(filepath, 'r') as f:
        raw_data = json.load(f)
    
    # Se c'è la chiave "instances", usala. Altrimenti usa il raw_data direttamente.
    instances = raw_data.get('instances', raw_data)
    
    records = []
    
    for inst in instances:
        inst_id = inst.get('instance_id', 'unknown')
        green_metrics = inst.get('green_metrics', {})
        
        # Iteriamo su ogni test case dentro l'istanza
        for test_name, metrics in green_metrics.items():
            base = metrics.get('base', {})
            head = metrics.get('head', {})
            
            # Estraiamo le medie (mean)
            energy_base = base.get('total_energy_joules_mean', np.nan)
            energy_head = head.get('total_energy_joules_mean', np.nan)
            
            time_base = base.get('duration_seconds_mean', np.nan)
            time_head = head.get('duration_seconds_mean', np.nan)
            
            power_base = base.get('power_watts_mean', np.nan)
            power_head = head.get('power_watts_mean', np.nan)

            if not np.isnan(energy_base) and not np.isnan(energy_head):
                records.append({
                    'instance_id': inst_id,
                    'test_name': test_name,
                    'energy_base': energy_base,
                    'energy_head': energy_head,
                    'time_base': time_base,
                    'time_head': time_head,
                    'power_base': power_base,
                    'power_head': power_head
                })
    
    return pd.DataFrame(records)

# --- 1. ESECUZIONE ---
print("🚀 Analisi del Dataset Green...")
df = process_green_dataset(FILE_GREEN)

if df is None or df.empty:
    print("❌ Nessun dato valido estratto. Controlla il percorso o la struttura del JSON.")
    exit()

print(f"✅ Estratti {len(df)} test validi da {df['instance_id'].nunique()} istanze.")

# --- 2. CALCOLO MIGLIORAMENTI ---
# Percentuale di risparmio (positivo = meglio)
df['energy_saved_pct'] = ((df['energy_base'] - df['energy_head']) / df['energy_base']) * 100
df['time_saved_pct'] = ((df['time_base'] - df['time_head']) / df['time_base']) * 100

# Statistiche Descrittive
print("\n📊 --- STATISTICHE GENERALI ---")
print(df[['energy_saved_pct', 'time_saved_pct']].describe())

# --- 3. TEST STATISTICI ---
print("\n🔬 --- VALIDAZIONE STATISTICA (Wilcoxon) ---")
# Usiamo Wilcoxon perché i dati non sono necessariamente normali e sono appaiati
stat_en, p_en = stats.wilcoxon(df['energy_base'], df['energy_head'])
stat_tm, p_tm = stats.wilcoxon(df['time_base'], df['time_head'])

print(f"⚡ ENERGIA: p-value = {p_en:.2e} -> {'✅ DIFFERENZA SIGNIFICATIVA' if p_en < 0.05 else '❌ NESSUNA DIFFERENZA'}")
print(f"⏱️ TEMPO:   p-value = {p_tm:.2e} -> {'✅ DIFFERENZA SIGNIFICATIVA' if p_tm < 0.05 else '❌ NESSUNA DIFFERENZA'}")

# --- 4. GRAFICI ---
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# A. Distribuzione Risparmio Energetico
sns.histplot(df['energy_saved_pct'], kde=True, ax=axes[0], color='green', bins=30)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Distribuzione Risparmio Energetico (%)')
axes[0].set_xlabel('% Risparmio (Positivo = Miglioramento)')

# B. Scatter Plot: Tempo vs Energia
sns.scatterplot(data=df, x='time_saved_pct', y='energy_saved_pct', ax=axes[1], alpha=0.6)
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].axvline(0, color='gray', linestyle='--')
axes[1].set_title('Correlazione: Tempo vs Energia')
axes[1].set_xlabel('% Risparmio Tempo')
axes[1].set_ylabel('% Risparmio Energia')

# C. Potenza Media (Watt) Base vs Head
df_melt = df.melt(value_vars=['power_base', 'power_head'], var_name='Version', value_name='Watts')
sns.boxplot(data=df_melt, x='Version', y='Watts', ax=axes[2], palette=["gray", "lightgreen"])
axes[2].set_title('Confronto Potenza Media (Watt)')

plt.tight_layout()
filename = "green_analysis_results.png"
plt.savefig(filename)
print(f"\n🖼️ Grafici salvati in: {filename}")
plt.show()

🚀 Analisi del Dataset Green...
❌ File non trovato: data/processed/green/swe_perf_green_k1.json
❌ Nessun dato valido estratto. Controlla il percorso o la struttura del JSON.


TypeError: object of type 'NoneType' has no len()

: 